# 卷积神经网络（CNN）：Conv2d 最小可运行实操

当前教学深度：intro。按“预测 → 运行 → 核对公式 → 修改参数”完成。

本 Notebook 不训练完整 CNN；它先解决图像张量、卷积参数和 shape 三个前置问题。


In [ ]:
import torch
from torch import nn

torch.manual_seed(7)


## 1. 先确认 NCHW

PyTorch 的图像输入为 [batch, channel, height, width]。不要把 NHWC 直接送入 Conv2d。


In [ ]:
x = torch.randn(2, 3, 32, 32)  # [batch, channel, height, width]
conv_same = nn.Conv2d(3, 8, kernel_size=3, stride=1, padding=1)
y_same = conv_same(x)
assert y_same.shape == (2, 8, 32, 32)
print('same padding:', tuple(x.shape), '->', tuple(y_same.shape))


## 2. 公式与实际输出必须一致

先用函数计算，再由 Conv2d 的输出 shape 交叉验证。这里专门防止漏写 2×padding。


In [ ]:
def output_size(size, kernel, padding, stride):
    return (size + 2 * padding - kernel) // stride + 1

assert output_size(32, 3, 1, 1) == 32
assert output_size(32, 3, 1, 2) == 16
print('formula checks:', output_size(32, 3, 1, 1), output_size(32, 3, 1, 2))


## 2.5 手算一个窗口

先看单通道、2×2 卷积核的一个位置，理解“逐元素相乘再求和”。


In [ ]:
small = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]])
kernel = torch.tensor([[1.0, 0.0], [0.0, -1.0]])
manual_first = (small[:2, :2] * kernel).sum()
assert manual_first.item() == -4.0
print('first window correlation result:', manual_first.item())


In [ ]:
conv_stride = nn.Conv2d(3, 8, kernel_size=3, stride=2, padding=1)
y_stride = conv_stride(x)
assert y_stride.shape == (2, 8, 16, 16)
parameter_count = sum(
    parameter.numel() for parameter in conv_stride.parameters()
)
print('stride=2:', tuple(y_stride.shape), 'parameters:', parameter_count)


## 3. 通道数改变与空间尺寸改变是两件事

out_channels 改变输出特征图数量；stride/padding 改变空间大小。下面将两者分开验证。


In [ ]:
conv_channels = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
y_channels = conv_channels(x)
assert y_channels.shape == (2, 16, 32, 32)
assert sum(parameter.numel() for parameter in conv_channels.parameters()) == 448
print('channels changed:', tuple(y_channels.shape))


## 4. 卷积后接池化

池化会缩小空间尺寸，但没有可训练卷积核。请与 stride=2 的卷积结果比较。


In [ ]:
pool = nn.MaxPool2d(kernel_size=2, stride=2)
pooled = pool(y_same)
assert pooled.shape == (2, 8, 16, 16)
print('pooling:', tuple(y_same.shape), '->', tuple(pooled.shape))


## 5. 调试练习

将 padding 改为 0：先预测 32 会变成多少，再修改代码和断言。若报通道错误，先打印 x.shape，而不是随意改 in_channels。


## 6. 提交前自检

1. 能写出 NCHW 的含义；2. 能手算两组输出尺寸；3. 能解释 224 个参数来自哪里；4. 能区分卷积与池化。完成后再进入分层测验。
